In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset
from fundus_vessels_toolkit.segment_to_graph.models.digraph_model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.models.trainer import DigraphGNNTrainer
from fundus_vessels_toolkit.utils.tree import tree_connected_components

vscode_theme()


In [ ]:
import datetime

PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]
opts = dict(resize_to=1024, root="tmp/DATA", ignore_recent=datetime.datetime(2026, 2, 19))
dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, **opts)
train_set, val_set, test_set = dataset.split_loaders(train_ratio=0.7, val_ratio=0.15)

## Visualize result from pred table


In [ ]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


### Load model from checkpoint


In [ ]:
model = DigraphGNNTrainer.load_from_checkpoint("tmp/last_mixed.ckpt").model.cuda().eval()


In [ ]:
ID = 16
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(test_set.get(ID).cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

digraph, _, od_yx, _ = test_set.get_sample(ID)
valid_branch = ~digraph.missing_branch()
av_gt = digraph.branch_av() <= 1
print(((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())
print(((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())


In [ ]:
torch.stack([out1.lines_logit, out2.lines_logit], dim=-1)

In [ ]:
m, pred_tree = test_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
)
m

In [ ]:
out.to_digraph().solve_optimal_arboresence(detect_major_av_error=True)

In [ ]:
B_ID = 144
pred_av[B_ID], out.av_logit[B_ID], pred_parent[B_ID], out.max_parent()[B_ID]

In [ ]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_by_branch(b1=164, sort_by_p=True).round(3)

In [ ]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []
node_ratio = []
branch_ratio = []

eval_set = test_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        digraph, _, od_yx, _ = eval_set.get_sample(i)
        assert digraph.graph is not None, "Graph must be loaded to infer tree"

        node_ratio += [digraph.graph.node_count / eval_set.graphs[i].node_count]
        branch_ratio += [digraph.graph.branch_count / eval_set.graphs[i].branch_count]

        valid_branch = ~digraph.missing_branch()
        av_gt = (digraph.branch_av() <= 1)[valid_branch]
        od = Point.parse(od_yx)

        art_branch = digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(eval_set.get(i).cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

In [ ]:
np.mean(node_ratio), np.mean(branch_ratio)

In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

In [ ]:
np.array(opti_av_acc)[21], np.array(pred_av_acc)[21]

In [ ]:
(np.array(pred_av_acc) - np.array(opti_av_acc)).argsort()[::-1]

In [ ]:
name